# Agentic Finance Evidence Gate

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crowdvector/polybridge-cookbooks/blob/main/agentic-finance/agentic-finance.ipynb)

This notebook runs the offline Agentic Finance Evidence Gate cookbook. It uses sanitized fixtures only, creates a normalized EvidencePacket, applies a deterministic gate, writes a decision memo and redacted JSONL audit log, and creates an Alpaca-style paper-preview object only if the gate clears.

This is research/demo software, not financial advice. It does not place trades and does not call live PolyBridge or broker APIs.


In [ ]:
import pathlib
import subprocess
import sys
import urllib.request

BASE_URL = "https://raw.githubusercontent.com/crowdvector/polybridge-cookbooks/main/agentic-finance"
REQ_PATH = pathlib.Path("requirements.txt")

if not REQ_PATH.exists():
    urllib.request.urlretrieve(f"{BASE_URL}/requirements.txt", REQ_PATH)

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(REQ_PATH)])


In [ ]:
import pathlib
import urllib.request

FILES = [
    "evidence_gate.py",
    "agentic_finance/__init__.py",
    "agentic_finance/cli.py",
    "agentic_finance/models.py",
    "agentic_finance/evidence.py",
    "agentic_finance/gate.py",
    "agentic_finance/memo.py",
    "agentic_finance/audit.py",
    "agentic_finance/redaction.py",
    "agentic_finance/brokers/__init__.py",
    "agentic_finance/brokers/alpaca.py",
    "fixtures/thesis.json",
    "fixtures/polybridge_search.response.json",
    "fixtures/polybridge_forecast.response.json",
]

for relative in FILES:
    path = pathlib.Path(relative)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{BASE_URL}/{relative}", path)


In [ ]:
import importlib.util
import pathlib

HELPER_PATH = pathlib.Path("evidence_gate.py")
spec = importlib.util.spec_from_file_location("evidence_gate", HELPER_PATH)
evidence_gate = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(evidence_gate)

result = evidence_gate.run_offline()
print("Decision:", result["gate_decision"].decision)
for label, path in result["paths"].items():
    print(label, path)


In [ ]:
from pathlib import Path

memo_path = result["paths"]["decision_memo"]
Path(memo_path).read_text(encoding="utf-8")
